# 01 · Tracing the antonym circuit in English and French

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/methuliH/multilingual-circuits-si-ta/blob/main/notebooks/01_trace_en_fr.ipynb)

**Goal:** replicate the multilingual-circuits finding from *On the Biology of a Large Language Model* on **Gemma-2-2B**, using the original prompts:

| Language | Prompt | Expected |
|---|---|---|
| English | `The opposite of "small" is "` | big / large |
| French | `Le contraire de "petit" est "` | grand |

**Steps**
1. Check the model answers both prompts correctly
2. Build an attribution graph for each prompt and record its replacement and completeness scores
3. View the graphs in the interactive viewer
4. Measure per-layer feature overlap between English and French, against unrelated-prompt baselines
5. Suppress the shared middle-layer features and check whether **both** languages lose the answer

**Before running:** Runtime → Change runtime type → **T4 GPU**, and add your Hugging Face token under 🔑 Secrets as `HF_TOKEN` (with notebook access on).

## 0 · Setup

In [ ]:
#@title Install circuit-tracer and log in to Hugging Face
try:
    import google.colab
    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")

    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.graph import compute_graph_scores
from circuit_tracer.utils import create_graph_files
from circuit_tracer.utils.demo_utils import display_topk_token_predictions

In [ ]:
# "gemma" = Gemma Scope per-layer transcoders
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend="transformerlens"
)
display_topk_token_predictions = partial(display_topk_token_predictions, tokenizer=model.tokenizer)
N_LAYERS = getattr(getattr(model, "cfg", None), "n_layers", 26)  # Gemma-2-2B has 26 layers
print("Layers:", N_LAYERS)

## 1 · Prompts and sanity check

The **baseline** prompts end in the same `"` token but ask for something unrelated. They tell us how much overlap to expect from shared tokens and grammar alone.

In [ ]:
PROMPTS = {
    "en": 'The opposite of "small" is "',
    "fr": 'Le contraire de "petit" est "',
}
BASELINES = {
    "en_unrelated": 'My favourite film is called "',
    "fr_unrelated": 'Mon film préféré s\'appelle "',
}

def top_k(prompt, k=5):
    with torch.inference_mode():
        logits, _ = model.feature_intervention(prompt, [])
    probs = logits.reshape(-1, logits.shape[-1])[-1].float().softmax(-1)
    p, idx = probs.topk(k)
    return [(model.tokenizer.decode(i), round(v.item(), 3)) for v, i in zip(p, idx)]

for name, prompt in {**PROMPTS, **BASELINES}.items():
    tokens = [model.tokenizer.decode(t) for t in model.tokenizer(prompt).input_ids]
    print(f"{name:13s} tokens={tokens}")
    print(f"{'':13s} top-5={top_k(prompt)}\n")

## 2 · Attribution graphs

Each graph takes a few minutes on a T4. The scores tell you how much of the computation the transcoders explain:
- **Replacement score:** how much flows from input to output through features rather than error nodes
- **Completeness score:** how much of each node's input comes from non-error nodes

📝 Record both in the findings log. They're the baseline to compare Sinhala and Tamil against later.

In [ ]:
GRAPH_DIR = Path("graphs"); GRAPH_DIR.mkdir(exist_ok=True)
graphs, scores = {}, {}

for lang, prompt in PROMPTS.items():
    graph = attribute(
        prompt=prompt,
        model=model,
        max_n_logits=10,
        desired_logit_prob=0.95,
        max_feature_nodes=8192,
        batch_size=256,
        offload="disk" if IN_COLAB else "cpu",
        verbose=True,
    )
    path = GRAPH_DIR / f"antonym_{lang}.pt"
    graph.to_pt(path)
    graphs[lang] = path
    scores[lang] = compute_graph_scores(graph)
    del graph
    torch.cuda.empty_cache()

for lang, (replacement, completeness) in scores.items():
    print(f"{lang}: replacement={replacement:.3f}  completeness={completeness:.3f}")

## 3 · View the graphs

Pick a graph from the dropdown at the top of the viewer. Click the top output node, follow its strongest inputs downward, and look for:
- an **antonym / opposite** feature on the final token (middle layers)
- a **small / size** feature on the `small` / `petit` token
- **say big / say grand** features near the top

In [ ]:
for lang, path in graphs.items():
    create_graph_files(
        graph_or_path=path,
        slug=f"antonym-{lang}",
        output_path="./graph_files",
        node_threshold=0.8,
        edge_threshold=0.98,
    )

from circuit_tracer.frontend.local_server import serve
PORT = 8046
server = serve(data_dir="./graph_files/", port=PORT)

if IN_COLAB:
    from google.colab import output as colab_output
    colab_output.serve_kernel_port_as_iframe(PORT, path="/index.html", height="800px", cache_in_notebook=True)
else:
    from IPython.display import IFrame
    display(IFrame(src=f"http://localhost:{PORT}/index.html", width="100%", height="800px"))

## 4 · Feature overlap by layer

Following the paper, we compare the **sets of active transcoder features** for two prompts at each layer using intersection-over-union (IoU), here at the final token. If English and French share a concept space, the antonym pair should overlap **more than the baselines**, especially in the **middle layers**.

In [ ]:
def final_token_features(prompt):
    """Return {layer: {feature_idx: activation}} for active features at the final token."""
    _, acts = model.get_activations(prompt, sparse=True)
    pos = acts.shape[1] - 1
    out = {layer: {} for layer in range(acts.shape[0])}
    if acts.is_sparse:
        acts = acts.coalesce()
        idx, vals = acts.indices(), acts.values()
        keep = (idx[1] == pos) & (vals > 0)
        for layer, feat, val in zip(idx[0][keep].tolist(), idx[2][keep].tolist(), vals[keep].float().tolist()):
            out[layer][feat] = val
    else:
        for layer in range(acts.shape[0]):
            nz = (acts[layer, pos] > 0).nonzero().flatten()
            out[layer] = {f: acts[layer, pos, f].item() for f in nz.tolist()}
    return out, pos

feats = {name: final_token_features(p) for name, p in {**PROMPTS, **BASELINES}.items()}

def iou_by_layer(a, b):
    fa, fb = feats[a][0], feats[b][0]
    return [len(fa[l].keys() & fb[l].keys()) / max(1, len(fa[l].keys() | fb[l].keys())) for l in range(N_LAYERS)]

pairs = {
    "EN antonym vs FR antonym": ("en", "fr"),
    "EN antonym vs FR unrelated (baseline)": ("en", "fr_unrelated"),
    "EN antonym vs EN unrelated (baseline)": ("en", "en_unrelated"),
}

Path("figures").mkdir(exist_ok=True)
plt.figure(figsize=(8, 4.5))
for label, (a, b) in pairs.items():
    plt.plot(range(N_LAYERS), iou_by_layer(a, b), marker="o", ms=3, label=label,
             linestyle="-" if "baseline" not in label else "--")
plt.xlabel("Layer"); plt.ylabel("Feature IoU at final token")
plt.title("Gemma-2-2B: shared features across languages")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("figures/01_overlap_en_fr.png", dpi=200)
plt.show()

### Shared middle-layer features

These features are active at the final token in **both** languages, ranked by their weaker activation. They're candidates for the language-independent **antonym** concept. Open the links to read their top activating examples on Neuronpedia.

In [ ]:
MIDDLE = range(N_LAYERS // 3, 2 * N_LAYERS // 3 + 1)
en, fr = feats["en"][0], feats["fr"][0]
shared = sorted(
    ((min(en[l][f], fr[l][f]), l, f) for l in MIDDLE for f in en[l].keys() & fr[l].keys()),
    reverse=True,
)
TOP_SHARED = [(l, f) for _, l, f in shared[:10]]
for score, l, f in shared[:10]:
    print(f"L{l:<2} #{f:<6} min-act={score:6.2f}  "
          f"https://www.neuronpedia.org/gemma-2-2b/{l}-gemmascope-transcoder-16k/{f}")

## 5 · Intervention: suppress the shared features

If these features carry the shared antonym concept, suppressing them should hurt the answer in **both** languages. Following the paper, we steer each feature to **−1× its original activation**.

Edit `FEATURES_TO_SUPPRESS` after you've inspected the features above. By default it uses the top shared ones.

In [ ]:
FEATURES_TO_SUPPRESS = TOP_SHARED[:5]   # list of (layer, feature_idx)
MULTIPLIER = -1.0

for lang, prompt in PROMPTS.items():
    lang_feats, pos = feats[lang]
    interventions = [
        (l, pos, f, MULTIPLIER * lang_feats[l][f])
        for l, f in FEATURES_TO_SUPPRESS if f in lang_feats[l]
    ]
    with torch.inference_mode():
        original_logits, _ = model.feature_intervention(prompt, [])
        new_logits, _ = model.feature_intervention(prompt, interventions)
    print(f"=== {lang}: {prompt}  ({len(interventions)} features suppressed)")
    display_topk_token_predictions(prompt, original_logits, new_logits)

## 6 · Record your findings

Add to `notes/` and the findings log:
- [ ] Top-5 predictions for each prompt (section 1)
- [ ] Replacement and completeness scores for each graph (section 2)
- [ ] Whether you found antonym, small/size and say-big features in the graphs (section 3)
- [ ] Whether EN–FR overlap beats both baselines, and in which layers (section 4); download `figures/01_overlap_en_fr.png`
- [ ] Which features you suppressed and how the predictions changed in each language (section 5)